In [ ]:
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [ ]:
class DataCurator():

    def __init__(self, df):
        self.seed = random.seed(52497)
        self.df = df
        self.height = 30
        self.width = 0
        self.end_point = 0
        self.n_pos = None
        self.data = None
        self.undersampled = None
        self.binned = None
        self.datalist = []
        self.past_heights = []

    def get_width(self):
        return self.width

    def get_col_names(self):
        return self.datalist

    def get_binned_data(self):
        return self.binned

    def set_height(self, n):
        self.height = n
        self.past_heights.append(self.height)
        
    def choose_height(self):
        self.height = random.randint(30, 300)
        self.past_heights.append(self.height)

    def shuffle_mat(self):
        np.random.shuffle(self.binned)

    def get_dataset(self):
        return self.data
    
    def gen_dataset(self):
        colset = self.df.columns
        tmpdf = self.df[colset[:self.df.shape[1]-1]].dropna(axis=0)
        print("Smoothing dataset...")
        for i in range(2,150,3):
            tmpdf[['xint_rolling_'+str(i), 'yint_rolling_'+str(i)]] = self.df[['xint','yint']].rolling(i).mean()
        tmpdf = tmpdf.dropna(axis=0)
        tmpdf = (tmpdf-tmpdf.min()) /(tmpdf.max()-tmpdf.min())
        self.datalist = tmpdf.columns
        self.width = tmpdf.shape[1]
        tmpdf['labels'] = self.df.loc[tmpdf.index, 'stim_onset']
        self.data = tmpdf
        self.end_point = tmpdf.shape[0]
        
        print('')

    def undersample(self):
        '''
        Takes in a binned matrix and the number of positive examples and balances
        the dataset thus that the number of negative examples are undersampled to 
        match the number of positive examples.
        '''
        tmpx = np.ones((self.n_pos*2, self.width+1, self.height))
        n_negative = 0
        j = 0
        for i in range(self.binned.shape[0]):
            if j == tmpx.shape[0]:
                # if we run out of tmpx slots, we stop
                continue

            if self.binned[i,-1,0] > 0:
                tmpx[j,:,:] = self.binned[i,:,:]
                j += 1
            elif self.binned[i,-1,0] == 0 & n_negative < self.n_pos:
                if random.uniform(0,1.0) < 0.35:
                    tmpx[j,:,:] = self.binned[i,:,:]
                    j += 1
                    n_negative += 0
            else:
                continue

        self.undersampled = tmpx


    def bin_data(self):
        tmpx = np.ones((self.end_point//self.height, self.width+1, self.height))
        self.n_pos = 0
        for i, t in enumerate(range(self.height,self.end_point+1,self.height)):
            tmpdf = self.data.iloc[t-self.height:t,:-1]
            tmpx[i,:self.width,:] = np.transpose(tmpdf.values)
            tmpx[i,self.width,:] = np.sum(self.data['labels'].loc[tmpdf.index])/self.height
            if tmpx[i,self.width,0] > 0.0:
                self.n_pos += 1
        np.random.shuffle(tmpx)
        # tmp1 = tmpx
        # np.random.shuffle(tmpx)
        # tmp2 = tmpx
        # np.random.shuffle(tmpx)
        # tmp3 = tmpx
        # tmpx = np.vstack((tmp1,tmp2,tmp3))
        self.binned = tmpx

    def gen_array(self):
        #self.choose_height()
        self.gen_dataset()
        self.bin_data()
            
    def get_processed_data(self):
        if len(self.binned) == 0:
            print("No binned arrays!")
        elif len(self.datalist) == 0:
            print("No column names!")
        elif type(self.undersampled) == type(None):
            return self.binned
        else:
            return self.undersampled

In [ ]:
bdf = pd.read_csv("./bdf_9.csv", index_col=0, usecols=['e_time', 'xint', 'yint', 'xint_diff', 'yint_diff', 'xint_diff_2', 'yint_diff_2', 'stim_onset'], low_memory=True, dtype={'xint':np.float32, 'yint':np.float32, 'xint_diff':np.float32, 'yint_diff':np.float32, 'xint_diff_2':np.float32, 'yint_diff_2':np.float32, 'stim_onset':np.int16})


In [ ]:
datamaster = DataCurator(bdf)
datamaster.gen_dataset()
data = datamaster.get_dataset()

In [ ]:
datamaster.get_col_names()

In [ ]:
pair_plot_list = [x for x in data.columns if x[0]=='x']

In [ ]:
data.head()

In [ ]:
sns.pairplot(data.drop('labels', axis = 1).iloc[:,:6])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.iloc[:,:-1], data.iloc[:,-1], test_size=0.4, random_state=705)

feature_extractor = RandomForestClassifier(random_state=873, n_jobs=10, verbose=1, warm_start=True)

In [ ]:
X_train.shape

In [ ]:
feature_extractor.fit(X_train, y_train)

In [ ]:
importances = feature_extractor.feature_importances_

In [ ]:
std = np.std([tree.feature_importances_ for tree in feature_extractor.estimators_], axis=0)

In [ ]:
gaze_freq_importances = pd.Series(importances, index=datamaster.get_col_names())

In [ ]:
fig, ax = plt.subplots()
ax.bar(range(30), gaze_freq_importances.iloc[:30].values, yerr=std[:30])
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots()
ax.bar(range(30), gaze_freq_importances.iloc[30:60].values, yerr=std[30:60])
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots()
ax.bar(range(30), gaze_freq_importances.iloc[60:90].values, yerr=std[60:90])
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

In [ ]:
gaze_freq_importances.iloc[:30]

In [ ]:
gaze_freq_importances.iloc[30:60]

In [ ]:
gaze_freq_importances.iloc[60:90]

In [ ]:
subset = gaze_freq_importances[gaze_freq_importances>0.01].index

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.loc[:,subset], data.iloc[:,-1], test_size=0.4, random_state=705)

In [ ]:
feature_extractor = RandomForestClassifier(random_state=873, n_jobs=10, verbose=1, warm_start=True)

In [ ]:
feature_extractor.fit(X_train, y_train)

In [ ]:
importances = feature_extractor.feature_importances_

In [ ]:
std = np.std([tree.feature_importances_ for tree in feature_extractor.estimators_], axis=0)

In [ ]:
gaze_freq_importances = pd.Series(importances, index=subset)

In [ ]:
fig, ax = plt.subplots()
ax.bar(range(gaze_freq_importances.shape[0]), gaze_freq_importances.values, yerr=std[:30])
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

In [ ]:
gaze_freq_importances.index

In [ ]:
data.to_csv('./bdf9_w_rollingavg.csv')

In [ ]:
yh = feature_extractor.predict(X_test)

In [ ]:
from sklearn import metrics  
print()
  
# using metrics module for accuracy calculation
print("ACCURACY OF THE MODEL: ", metrics.accuracy_score(y_test, yh))

In [ ]:
len(yh)

In [ ]:
len(yh[yh==1])

In [ ]:
len(y_test[y_test==1])